In [1]:
import tempfile

from api import RayfishApp
from api.pydantic_models import ParametersInput, SimulationConfig

sim_config = SimulationConfig(
    light_type="incoherent",
    estimation_parameter="distance_2_sources",
    simulation_dimension=1,
)
param_config = ParametersInput(
    wave_vector=2.0,
    rayleigh_length=1.0,
    number_of_point_source_distributions=20,
    maximal_length=4.0,
    minimal_spacing=0.1,
)

with tempfile.TemporaryDirectory() as temp_dir:
    app = RayfishApp(
        simulation_config=sim_config, input_parameters=param_config, save_dir=temp_dir
    )
    result = app.run_simulation()
    result["quantum"].name_parameters({
        "quantum_var_0_0": "var_xy",
        "quantum_var_1_1": "var_z",
        "quantum_cov_0_1": "cov_xyz",
        "quantum_cov_1_0": "cov_xyz_2",
    })
    fig = result["quantum"].generate_figure()
    fig.show()

## Plotting the results

In [4]:
import numpy as np
import pandas as pd
import plotly.express as px

qfi = np.real(result["covariance_matrix_for_entire_distribution"])
gamma = np.imag(result["covariance_matrix_for_entire_distribution"])
x = result["spacing"]
dfs = {
    "spacing": x,
    "var_xy": qfi[:, 0, 0],
    "var_z": qfi[:, 1, 1],
    "cov_xyz_1": qfi[:, 0, 1],
    "cov_xyz_2": qfi[:, 1, 0],
    "gamma_xy": gamma[:, 0, 0],
    "gamma_z": gamma[:, 1, 1],
    "cov_gamma_1": gamma[:, 0, 1],
    "cov_gamma_2": gamma[:, 1, 0],
}

df = pd.DataFrame.from_dict(dfs)
df = df.set_index("spacing")
sel_df = df  # df[["var_xy", "var_z"]]
fig = px.scatter(sel_df, x=sel_df.index, y=sel_df.columns)
fig.show()